In [ ]:
!pip install datasets transformers sentence-transformers torch scikit-learn -q

In [ ]:
from datasets import load_dataset

# Load CSV using Hugging Face datasets
dataset = load_dataset(
    "csv",
    data_files="train.csv"
)

train_ds = dataset["train"]

print(train_ds)
print(train_ds.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [ ]:
def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example


train_ds = train_ds.map(combine)

answer1 = len(train_ds[51]["combined_text"])

print("Q1 Answer =", answer1)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1 Answer = 614


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
answer2 = tokenizer.vocab_size

print("Q2 Answer =", answer2)

Q2 Answer = 30522


In [ ]:
sep_id = tokenizer.sep_token_id

print("Q3 Answer =", sep_id)

Q3 Answer = 102


In [ ]:
tokens = tokenizer(
    list(train_ds["prompt"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

answer4 = tokens["input_ids"].shape

print("Q4 Answer =", answer4)

Q4 Answer = torch.Size([2000, 128])


In [ ]:
hidden_size = 768
num_heads = 12

answer5 = hidden_size // num_heads

print("Q5 Answer =", answer5)

Q5 Answer = 64


In [ ]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained(
    "bert-base-uncased"
)

text = train_ds[0]["prompt"]

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)


last_hidden = outputs.last_hidden_state

answer6 = last_hidden.shape

print("Q6 Answer =", answer6)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q6 Answer = torch.Size([1, 31, 768])


In [ ]:
cls_vector = last_hidden[0,0,:]

answer7 = cls_vector[:5].sum().item()

print(
    "Q7 Answer =",
    round(answer7,4)
)

Q7 Answer = -1.2001


In [ ]:
model_att = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

sentence = "Light-ion fusion is a technique."

inputs = tokenizer(
    sentence,
    return_tensors="pt"
)


with torch.no_grad():
    outputs = model_att(**inputs)


tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

# Extract attention matrix for the last layer and first head
# outputs.attentions is a tuple, last layer is at index -1
# Shape: (batch_size, num_heads, sequence_length, sequence_length)
last_layer_attention = outputs.attentions[-1]
first_head_attention = last_layer_attention[0, 0, :, :]

# Find the index of the 'fusion' token
try:
    fusion_token_index = tokens.index('fusion')
except ValueError:
    # Handle cases where 'fusion' might be tokenized differently (e.g., 'fu', '##sion')
    # For this specific sentence, 'fusion' is a single token as seen in previous output
    print("'fusion' token not found as a single token. Please check token list.")
    fusion_token_index = -1 # Or handle appropriately

# [CLS] token is always at index 0
cls_token_index = 0

# Get the attention weight from [CLS] to 'fusion'
if fusion_token_index != -1:
    attention_weight = first_head_attention[cls_token_index, fusion_token_index].item()
    answer_att = round(attention_weight, 4)
    print("Q8 Answer =", answer_att)
else:
    print("Cannot calculate Q8 Answer as 'fusion' token was not found.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q8 Answer = 0.1025


In [ ]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


prompt_emb = st_model.encode(
    train_ds[0]["prompt"],
    convert_to_tensor=True
)

b_emb = st_model.encode(
    train_ds[0]["B"],
    convert_to_tensor=True
)


similarity = util.cos_sim(
    prompt_emb,
    b_emb
)


answer9 = similarity.item()

print(
    "Q9 Answer =",
    round(answer9,4)
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9 Answer = 0.7658


In [ ]:
def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    if actual in predicted:
        return 1/(predicted.index(actual)+1)

    return 0


def map3(actuals, predictions):

    scores=[]

    for a,p in zip(actuals,predictions):
        scores.append(
            apk(a,p,3)
        )

    return sum(scores)/len(scores)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


tfidf_predictions=[]


for row in train_ds:

    texts=[
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    vec=TfidfVectorizer()

    mat=vec.fit_transform(texts)

    sims=cosine_similarity(
        mat[0],
        mat[1:]
    )[0]

    labels=["A","B","C","D","E"]

    top3=[
        labels[i]
        for i in sims.argsort()[::-1][:3]
    ]

    tfidf_predictions.append(top3)

In [ ]:
minilm_predictions=[]


for row in train_ds:

    prompt=row["prompt"]

    options=[
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]


    p_emb=st_model.encode(
        prompt,
        convert_to_tensor=True
    )


    o_emb=st_model.encode(
        options,
        convert_to_tensor=True
    )


    scores=util.cos_sim(
        p_emb,
        o_emb
    )[0]


    labels=["A","B","C","D","E"]

    top3=[
        labels[i]
        for i in scores.argsort(descending=True)[:3]
    ]

    minilm_predictions.append(top3)

In [ ]:
answers = train_ds["answer"]

count=0


for ans,t,m in zip(
    answers,
    tfidf_predictions,
    minilm_predictions
):

    if ans not in t and ans in m:
        count += 1


print(
    "Improvement Count =",
    count
)

Improvement Count = 564


In [ ]:
from transformers import pipeline


classifier = pipeline(
    "zero-shot-classification"
)


candidate_labels=[
    train_ds[1]["A"],
    train_ds[1]["B"],
    train_ds[1]["C"]
]


result = classifier(
    train_ds[1]["prompt"],
    candidate_labels
)


print(result)


answer11=result["scores"][0]

print(
    "Q11 Answer =",
    round(answer11,4)
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [ ]:
result_multi = classifier(
    train_ds[1]["prompt"],
    candidate_labels,
    multi_label=True
)


softmax_sum=sum(result["scores"])

sigmoid_sum=sum(result_multi["scores"])


answer12=abs(
    softmax_sum-sigmoid_sum
)


print(
    "Q12 Answer =",
    answer12
)

Q12 Answer = 0.999490372636501


In [ ]:
from transformers import pipeline


generator = pipeline(
    "text-generation", # Changed from "text2text-generation"
    model="google/flan-t5-small"
)


row=train_ds[0]


query = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    "Answer with just the letter A or B."
)


output = generator(
    query,
    max_new_tokens=5
)


print(output)


answer13 = output[0]["generated_text"]

print(
    "Q13 Answer =",
    answer13
)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

[{'generated_text': "Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B."}]
Q13 Answer = Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answe